This notebook preserves the inference-stage code of the published case study (Ma, Zhang & Song, 2025, Spatial Statistics 69, 100920).
Outputs have been cleared for release, and the author's original data/checkpoint paths were normalized to `./data/`, `./outputs/`, and `./checkpoints/`; the datasets and trained checkpoints themselves are not distributed with this repository. The reusable algorithm lives in the `gignn` package.

本 notebook 保留了已发表论文案例研究的推理阶段代码。输出已随发布清空，原始数据与检查点路径已规范化为 `./data/`、`./outputs/`、`./checkpoints/`；数据集与已训练检查点不随本仓库分发。可复用算法见 `gignn` 包。

栅格数据裁切

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np


In [ ]:
# 绘图函数
def set_axis_format(ax, x_interval=0.3, y_interval=0.15):
    """设置坐标轴的格式和刻度间隔"""
    ax.set_xlabel("Longitude")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: format_coord(x, pos, 'E' if x >= 0 else 'W')))
    ax.xaxis.set_major_locator(mticker.MultipleLocator(x_interval))

    ax.set_ylabel("Latitude")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, pos: format_coord(x, pos, 'N' if x >= 0 else 'S')))
    ax.yaxis.set_major_locator(mticker.MultipleLocator(y_interval))

def format_coord(coord, pos, hemisphere):
    """格式化坐标为度分秒格式"""
    deg = int(coord)
    min = int((coord - deg) * 60)
    sec = int((coord - deg - min/60) * 3600)
    return f"{abs(deg)}°{abs(min)}′{hemisphere}"#{abs(sec)}″{hemisphere}


```mermaid
graph TD
    Y[Y] --> D[Dataset]
    X[Xs] --> D[Dataset]
    


研究区

In [ ]:
# 潜在自变量与因变量

# 创建一个字典，将原始名称映射到文章中的变量名称
potential_indicators = {
    "Used at least one form of public transport (train, tram, bus, ferry) (no.)": "Number of residents work by public transport",
    "Travelled to work by car (as a driver or passenger) as at least one method of travel (no.)": "Number of residents drive to work",
    "Average household size (no. of persons)": "Average household size",
    "Average family size (no. of persons)": "Average family size",
    "Married (%)": "Percentage of married residents",
    "Median weekly household rental payment ($)": "Median-level house rental price",
    "House ownership (%)": "House ownership",
    "Total cars in that region (no.)": "Total vehicles",
    "Number of jobs": "Total jobs",
    "Number of employee jobs - manufacturing": "Number of residents working for manufacturing industry",
    "Number of employee jobs - construction": "Number of residents working for construction industry",
    "Number of employee jobs - finance and insurance services": "Number of residents working for finance industry",
    "Number of employee jobs - professional, scientific and technical services": "Number of residents working for high technology development",
    "Postgraduate degree (%)": "Tertiary education at master's level",
    "Bachelor degree (%)": "Tertiary education at bachelor's level",
    "Mean employee income ($)": "Mean employee income",
    "Mean own unincorporated business income ($)": "Mean own unincorporated business income",
    "Mean investment income ($)": "Mean investment income",
    "Total income (excl. Government pensions and allowances) - gini coefficient": "Gini coefficient of total income",
    "$3000 or more per week (%)": "Percentage of high income residents",
    "ANLI": "Urbanization level"
}

selected_features = [
    "Tertiary education at bachelor's level",
    'Mean investment income',
    'Mean employee income',
    'Percentage of married residents',
    'Gini coefficient of total income'
 ]

ctrl_X_names = ['Average family size',
 'Number of residents working for finance industry',
 'Average household size',
 'Percentage of married residents',
 'Number of residents work by public transport']

tmp = {}
ctrl_potential_indicators = {}
for i in potential_indicators.keys():
    if potential_indicators[i] in selected_features:
        tmp[i] = potential_indicators[i]
    if potential_indicators[i] in ctrl_X_names:
        ctrl_potential_indicators[i] = potential_indicators[i]

potential_indicators = tmp

In [ ]:
report = {
    "year": [],
    "city": [],
    "accuracy": []
}
for i in selected_features:
    report[i] = []


In [ ]:

ctrl_report = {
    "year": [],
    "city": [],
    "accuracy": []
}
for i in ctrl_X_names:
    ctrl_report[i] = []

In [ ]:
cities = ["Darwin", "Bunbury", "Adelaide","Perth"]
CITY = cities[-1]
YEAR = 2021
report["year"].append(YEAR)
report["city"].append(CITY)
ctrl_report["year"].append(YEAR)
ctrl_report["city"].append(CITY)


In [ ]:
# 加载不同尺度的研究区
SA2 = gpd.read_file(f'./data/{CITY}_SA2.shp').to_crs(3857)[["SA2_CODE21", "geometry"]]
print(f"{CITY} ({YEAR}): {len(SA2)}")

In [ ]:
influence_system = {}
ctrl_influence_system = {}
SA2_Xs_Y = SA2.copy()

In [ ]:
# 缺失数据处理
def fill_missing_data(df, input_year, exam_column):
    # Creating a copy of the dataframe to avoid modifying the original one
    df_filled = df.copy()
    year = input_year

    while year > 2016:
        # Filter dataframe for the specific year
        df_year = df[df['Year'] == year]
        df_year_column = df[df['Year'] == year][exam_column]
        # Calculate the percentage of missing values represented by "-"
        missing_percentage = pd.merge(SA2,df_year,left_on="SA2_CODE21",right_on="Code")[exam_column].isin(["-"]).sum() / len(SA2)
        complete_percentage = round(1-missing_percentage, 4)*100
        # Check if missing values percentage is more than 60%
        if complete_percentage < 80:
            # If more than 60% missing values found, go to the previous year
            print(f"正在检查{year}-{exam_column}完整性{complete_percentage}%")
            year -= 1
            # print(f"尝试重新检查{year}-{exam_column}")
        else:
            # If no or acceptable amount of missing values, replace the corresponding rows in df_filled
            df_filled.loc[df_filled['Year'] == input_year, exam_column] = df_year_column.fillna(0).apply(convert_to_float).tolist()
            print(f"{input_year}年{exam_column}数据由{year}年数据代替，数据完整率为{complete_percentage}%")
            break

    return df_filled, year

In [ ]:
# 转换函数：去除逗号并转换为浮点数
def convert_to_float(x):
    try:
        return float(str(x).replace(',', ''))
    except ValueError:
        return 0  # 或者根据需要选择其他的处理方式


社区信息导入

In [ ]:
community_census = pd.read_csv("./data/all_community_census.csv")
community_census["Code"] = community_census["Code"].astype(str)
community_census["Year"] = community_census["Year"].astype(int)
for c in potential_indicators.keys():
    if c in community_census.columns:
        community_census,y = fill_missing_data(community_census,input_year=YEAR, exam_column=c)
        report[potential_indicators[c]].append(y)
community_census = community_census[community_census["Year"] == YEAR]
SA2_community_merge = pd.merge(SA2,community_census,left_on="SA2_CODE21",right_on="Code")
SA2_community = SA2_community_merge[SA2_community_merge.columns[:2]].copy()
for c in SA2_community_merge.columns[2:]:
    if c in potential_indicators.keys():
        SA2_community.loc[:, potential_indicators[c]] = SA2_community_merge[c].fillna(0).apply(convert_to_float)
# SA2_community.loc[:, "Total cars in that region (no.)"] = SA2_community.apply(lambda x: (x["One motor vehicle (no.)"] if ) + 2 * x["Two motor vehicles (no.)"] + 3 * x["Three motor vehicles (no.)"] + 4 * x["Four or more motor vehicles (no.)"], axis=1)
for c in SA2_community.columns:
    if "motor" in c:
        del SA2_community[c]
influence_system["community"] = SA2_community.columns.tolist()[2:] 

In [ ]:
def test_invalid_data():
    tmp = community_census = pd.read_csv("./data/all_community_census.csv")
    tmp["Code"] = tmp["Code"].astype(str)
    tmp["Year"] = tmp["Year"].astype(int)
    tmp_2016 = tmp[tmp["Year"] == 2016]
    tmp_2021 = tmp[tmp["Year"] == 2021]
    tmp_com = pd.DataFrame({
        "id": SA2["SA2_CODE21"].tolist(),
        "Married (%) - 2016":pd.merge(SA2,tmp_2016,left_on="SA2_CODE21",right_on="Code")["Married (%)"].fillna(0).apply(convert_to_float).tolist(),
        "Married (%) - 2021":pd.merge(SA2,tmp_2021,left_on="SA2_CODE21",right_on="Code")["Married (%)"].fillna(0).apply(convert_to_float).tolist()
    } )
    tmp_com.loc[:,"Married (%) - 2016 - Class"] = [j for j in mc.NaturalBreaks(tmp_com["Married (%) - 2016"], k=5).yb]
    tmp_com.loc[:,"Married (%) - 2021 - Class"] = [j for j in mc.NaturalBreaks(tmp_com["Married (%) - 2021"], k=5).yb]
    tmp_com.loc[:,"differ"] = tmp_com["Married (%) - 2021"] - tmp_com["Married (%) - 2016"]
    tmp_com

雇员信息

In [ ]:
employ_census = pd.read_csv("./data/all_employ_census.csv")
employ_census["Code"] = employ_census["Code"].astype(str)
employ_census["Year"] = employ_census["Year"].astype(int)
for c in potential_indicators.keys():
    if c in employ_census.columns:
        employ_census,y = fill_missing_data(employ_census,input_year=YEAR, exam_column=c)
        report[potential_indicators[c]].append(y)
employ_census = employ_census[employ_census["Year"] == YEAR]
SA2_employ_merge = pd.merge(SA2,employ_census,left_on="SA2_CODE21",right_on="Code")
SA2_employ = SA2_employ_merge[SA2_employ_merge.columns[:2]].copy()
for c in SA2_employ_merge.columns[2:]:
    if c in potential_indicators.keys():
        SA2_employ.loc[:, potential_indicators[c]] = SA2_employ_merge[c].fillna(0).apply(convert_to_float)
influence_system["employ"] = SA2_employ.columns.tolist()[2:]

收入信息

In [ ]:
income_census = pd.read_csv("./data/all_income_census.csv")
income_census["Code"] = income_census["Code"].astype(str)
income_census["Year"] = income_census["Year"].astype(int)
for c in potential_indicators.keys():
    if c in income_census.columns:
        income_census,y = fill_missing_data(income_census,input_year=YEAR, exam_column=c)
        report[potential_indicators[c]].append(y)
income_census = income_census[income_census["Year"] == YEAR]
SA2_income_merge = pd.merge(SA2,income_census,left_on="SA2_CODE21",right_on="Code")#.fillna(0)
SA2_income = SA2_income_merge[SA2_income_merge.columns[:2]].copy()
for c in SA2_income_merge.columns[2:]:
    if c in potential_indicators.keys():
        SA2_income.loc[:, potential_indicators[c]] = SA2_income_merge[c].fillna(0).apply(convert_to_float)
influence_system["income"] = SA2_income.columns.tolist()[2:]

In [ ]:
for k in influence_system.keys():
    print("【{}】 \n{}".format(k, ", \n".join(influence_system[k])))

hi 在SA2级别下 雇员、社区、收入 与行政范围矢量数据连接较为完整 但是具体指标的缺失值问题比较明显 我们暂时可以以缺失值小于5%作为阈值 筛选出一些数据完整度较高的指标

In [ ]:
# 分区统计代码
import rasterio as rio
class ZonalStatistics(object):
    def __init__(self, raster_path, shp_path) -> None:
        self.raster_path = raster_path
        if "shp" in shp_path:
            self.gdf = gpd.read_file(shp_path)
        else:
            self.gdf = shp_path.to_crs(4326)
        self._init_params(self.raster_path)
        
    def _init_params(self, raster_path):
        src = rio.open(raster_path)
        self.transform = src.transform
        self.crs = src.crs
        self.shape = src.shape
    
    def _geometry_mask(self, src, geometries, all_touched=False):
        if isinstance(src, rio.DatasetReader):
            pass
        elif isinstance(src, str):
            src = rio.open(src)
        else:
            raise ValueError

        if not isinstance(geometries, (tuple, list)):
            raise ValueError
        from rasterio import features
        geometry_mask = features.geometry_mask(
            geometries=geometries,
            out_shape=src.shape,
            transform=src.transform,
            all_touched=all_touched,
            invert=True)
        
        return geometry_mask
    
    def _valid_range(self, mask):
        mask_col = np.any(mask, axis=0)
        mask_row = np.any(mask, axis=1)
        col_index = np.array(np.where(mask_col)[0])
        row_index = np.array(np.where(mask_row)[0])

        if col_index.size == 0 or row_index.size == 0:
            return None, None  # 返回None表示没有有效范围

        min_col, max_col = col_index.min(), col_index.max()#min(col_index), max(col_index)
        min_row, max_row = row_index.min(), row_index.max()#min(row_index), max(row_index)

        return (min_row, max_row), (min_col, max_col)
    
    def _read_from_geometry(self, geometries, all_touched=False):

        src = rio.open(self.raster_path)
        mask = self._geometry_mask(src, geometries, all_touched)
        valid_range = self._valid_range(mask)

        if valid_range == (None, None):
            return None  # 返回None表示没有有效数据

        (min_row, max_row), (min_col, max_col) = valid_range
        from rasterio.windows import Window
        window = Window.from_slices(rows=(min_row, max_row+1), 
                                    cols=(min_col, max_col+1))

        geom_array = src.read(1, window=window)
        
        geom_mask = ~mask[min_row:max_row+1, min_col:max_col+1]
        nodata_mask = (geom_array == src.nodata)
        nan_mask = np.isnan(geom_array)
        geom_array = np.ma.masked_array(geom_array, 
                                        geom_mask | nodata_mask | nan_mask)

        return geom_array
    
    def _statistics(self, geom_array, statistics_type):
        if statistics_type == 'min':
            return np.ma.min(geom_array)
        elif statistics_type == 'max':
            return np.ma.max(geom_array)
        elif statistics_type == 'median':
            return np.ma.median(geom_array)
        elif statistics_type == 'sum':
            return np.ma.sum(geom_array)
        elif statistics_type == 'std':
            return np.ma.std(geom_array)
        elif statistics_type == 'range':
            return np.ma.max(geom_array)-np.ma.min(geom_array)
        elif statistics_type == 'mean':
            return np.ma.mean(geom_array)
        else:
            raise ValueError
    
    def zonal_statistics(self, zone_field, statistics_type=None, 
                         func=None, all_touched=False):
        
        for i, geom in enumerate(self.gdf.geometry.to_list()):
            
            geom_array = self._read_from_geometry([geom], all_touched)

            if geom_array is None:
                print(f"Geometry {i} does not overlap with raster data. {geom_array}")
                continue
            
            if (isinstance(func, type(None)) and 
                isinstance(statistics_type, type(None))):
                raise ValueError
            
            if isinstance(func, type(None)):
                value = self._statistics(geom_array, statistics_type)
            else:
                value = func(geom_array)
            
            if isinstance(value, type(np.ma.masked)):
                continue
            else:
                self.gdf.loc[i, zone_field] = value
        self.gdf.fillna(0,inplace=True)

In [ ]:
# Xs 属性挂载
SA2_Xs_Y = SA2.copy()
for c in influence_system.keys():
    for i in influence_system[c]:
        if i in selected_features:
            print(i)
            SA2_Xs_Y.loc[:, i] = eval("SA2_{}".format(c))[i]
# 分区统计
zs = ZonalStatistics(f'./data/VIIRS_Nighttime_{YEAR}_{CITY}.tif', SA2.to_crs(4326).copy())
zs.zonal_statistics(zone_field='NL', statistics_type='sum')
SA2_Xs_Y.loc[:, "Urbanization level"] = zs.gdf["NL"] / (SA2_Xs_Y.geometry.area/10**4)


规范数据集

In [ ]:
import pandas as pd
Y_name = "Urbanization level"
X_names = list(SA2_Xs_Y.columns[2:][:-1])
uni_id = "SA2_CODE21"
SA2_Xs_Y.fillna(0, inplace=True)
dataset = SA2_Xs_Y.copy()

地理探测器数据

In [ ]:
# 输出 x 与 Y
import mapclassify as mc
geo_df = dataset.copy()

In [ ]:
# 对 Xs 进行分层
for i in geo_df[[*X_names, Y_name]]:
    if i not in [Y_name]:
        try:
            geo_df[i] = [j for j in mc.NaturalBreaks(dataset[i], k=5).yb]
        except:
            print(i)
    elif i == Y_name:
        geo_df[i] = [j for j in mc.NaturalBreaks(dataset[i], k=3).yb]
# geo_df[[*X_names, Y_name]].to_csv(f"./data/{CITY}_x_Y.csv",index=None)

## 图神经网络构建

### 构建图

In [ ]:
# 空间关系概念化
try:
    print("数据集已加载，采样个数：{}".format(len(geo_df)))
except:
    print("数据失败")

# 使用 sjoin 计算相邻拓扑关系
geo_df.loc[:,"id"] = np.array(geo_df.index)
geo_df_buffer = gpd.GeoDataFrame({"id": geo_df["id"]}, geometry=geo_df.buffer(100))
# 使用 sjoin 计算相邻拓扑关系
neighbors = gpd.sjoin(geo_df, geo_df_buffer, predicate='intersects', lsuffix='left', rsuffix='right')

# 去除与自身相交的记录
neighbors = neighbors[neighbors['id_left'] != neighbors['id_right']][['id_left', 'id_right']]

# 创建邻接数组，不使用eval
neighbors['valid_arr'] = neighbors.apply(lambda x: [x['id_left'], x['id_right']], axis=1)

# 显示结果
neighbors

构建图

In [ ]:
# 导库并确定随机种子
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.nn.init as init
from torch_geometric.data import Data
import torch_geometric.utils as utils
from torch_geometric.nn import GATConv, SAGEConv, GINConv, Sequential, ChebConv, MessagePassing
from torch_geometric.utils import add_self_loops, degree
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score, f1_score
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"

def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

rand_seed = 42
set_random_seed(rand_seed)


In [ ]:
# 地理信息嵌入数据集

# 提取节点特征

# 对照组：特征使用24个未进行数据分层的环境变量
# 目标模型：使用分层后的数据 geo_df_discretization_id
feature_source = geo_df.copy()
feature_source["id"] = np.array(feature_source.index)
# 提取节点特征
node_features =feature_source[selected_features].values#, 'PRE' feature_source.columns[:-4]
node_features = torch.tensor(node_features, dtype=torch.float32)

# 对节点ID进行因子化，以得到从0开始的连续整数
id_mapping = pd.Series(range(len(feature_source["id"])), index=feature_source["id"])
valid_ids = feature_source["id"]
valid_neighbors = neighbors[(neighbors["id_left"].isin(valid_ids)) & (neighbors["id_right"].isin(valid_ids))]
id_mapping = pd.Series(range(len(valid_ids)), index=valid_ids)
valid_neighbors.loc[:, "id_left"] = id_mapping[valid_neighbors["id_left"]].values
valid_neighbors.loc[:, "id_right"] = id_mapping[valid_neighbors["id_right"]].values

# 提取边索引
edge_index = valid_neighbors[['id_left', 'id_right']].values.astype(int)
edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()

# 构建邻接字典
def build_adjacency_dict_from_edge_index(edge_index, num_nodes):
    adjacency_dict = {i: [] for i in range(num_nodes)}
    for src, dest in zip(edge_index[0], edge_index[1]):
        src = int(src.item()) 
        dest = int(dest.item())  
        adjacency_dict[src].append(dest)
        adjacency_dict[dest].append(src)  # 无向图需要两个方向都加入

    return adjacency_dict
    
adjacency_dict = build_adjacency_dict_from_edge_index(edge_index, len(feature_source))


In [ ]:
# 计算不同特征的高斯核函数的差异

def calculate_edge_attributes(node_features, edge_index, sigma=2.0):
    # 计算高斯核函数
    def gaussian_kernel_similarity(x1, x2, sigma=2.0):
        # 计算高斯核函数
        diff = x1 - x2
        sq_diff = torch.pow(diff, 2)
        return torch.exp(-sq_diff / (2 * sigma ** 2))
    # 初始化边属性
    num_edges = edge_index.size(1)
    num_features = node_features.size(1)
    edge_attributes = torch.zeros(num_edges, num_features)

    # 针对每个边计算每个特征的高斯核函数
    for i in range(num_edges):
        source_node = edge_index[0, i]
        target_node = edge_index[1, i]

        for j in range(num_features):
            source_feature = node_features[source_node, j]
            target_feature = node_features[target_node, j]
            edge_attributes[i, j] = gaussian_kernel_similarity(source_feature, target_feature, sigma)

    # 计算不同特征的高斯核函数的差异
    # 这里的“差异”可以根据你的需要进行定义
    # 例如，可以是特征核函数值的均方差，总和，最大值等
    edge_attr_diff = torch.var(edge_attributes, dim=1)

    return edge_attributes

In [ ]:
# 目标模型特征：由地理探测器的结果指示的地理变量（经过分层处理）构成
# 提取标签
target_y = feature_source[Y_name].values
target_y = torch.tensor(feature_source[Y_name].values, dtype=torch.long)

# 节点特征是x，边索引是edge_index
target_x, target_edge_index = node_features, edge_index
# 计算边属性
# 应用高斯核函数
diff = target_x[target_edge_index[0]] - target_x[target_edge_index[1]]
target_edge_attr = calculate_edge_attributes(target_x, target_edge_index)
# 将边属性添加到图数据对象中
target_data = Data(x=target_x, edge_index=target_edge_index, edge_attr=target_edge_attr, y=target_y)
target_data.adjacency_dict = adjacency_dict

def remove_isolated_nodes(adjacency_dict, features, labels):
    isolated_nodes = []

    # 遍历节点
    for node in adjacency_dict:
        if not adjacency_dict[node]:
            isolated_nodes.append(node)

    # 删除孤立节点
    for node in isolated_nodes:
        del adjacency_dict[node]

    # 重新构建邻居节点列表，删除对应孤立节点的出现
    for node in adjacency_dict:
        adjacency_dict[node] = [neighbor for neighbor in adjacency_dict[node] if neighbor not in isolated_nodes]

    # 删除特征矩阵中对应的孤立节点的数据
    features = np.delete(features, isolated_nodes, axis=0)

    # 删除标签中对应的孤立节点的数据
    labels = np.delete(labels, isolated_nodes, axis=0)

    return adjacency_dict, features, labels

# 调用函数来删除孤立节点
# data.adjacency_dict, data.x, data.y = remove_isolated_nodes(data.adjacency_dict, data.x, data.y)

In [ ]:
# 对照组特征：由未进行分层处理的全部地理变量构成

# community
ctrl_community_census = pd.read_csv("./data/all_community_census.csv")
ctrl_community_census["Code"] = ctrl_community_census["Code"].astype(str)
ctrl_community_census["Year"] = ctrl_community_census["Year"].astype(int)
ctrl_community_census.loc[:, "Total cars in that region (no.)"] = ctrl_community_census.apply(lambda x: (x["One motor vehicle (no.)"]) + 2 * x["Two motor vehicles (no.)"] + 3 * x["Three motor vehicles (no.)"] + 4 * x["Four or more motor vehicles (no.)"], axis=1)
for c in ctrl_community_census.columns:
    if "motor" in c:
        del ctrl_community_census[c]
for c in ctrl_potential_indicators.keys():
    if c in ctrl_community_census.columns:
        ctrl_community_census,y = fill_missing_data(ctrl_community_census,input_year=YEAR, exam_column=c)
        ctrl_report[ctrl_potential_indicators[c]].append(y)
ctrl_community_census = ctrl_community_census[ctrl_community_census["Year"] == YEAR]
ctrl_SA2_community_merge = pd.merge(SA2,ctrl_community_census,left_on="SA2_CODE21",right_on="Code")
ctrl_SA2_community = ctrl_SA2_community_merge[ctrl_SA2_community_merge.columns[:2]].copy()
for c in ctrl_SA2_community_merge.columns[2:]:
    if c in ctrl_potential_indicators.keys():
        ctrl_SA2_community.loc[:, ctrl_potential_indicators[c]] = ctrl_SA2_community_merge[c].fillna(0).apply(convert_to_float)
ctrl_influence_system["community"] = ctrl_SA2_community.columns.tolist()[2:] 

# employ
ctrl_employ_census = pd.read_csv("./data/all_employ_census.csv")
ctrl_employ_census["Code"] = ctrl_employ_census["Code"].astype(str)
ctrl_employ_census["Year"] = ctrl_employ_census["Year"].astype(int)
for c in ctrl_potential_indicators.keys():
    if c in ctrl_employ_census.columns:
        ctrl_employ_census,y = fill_missing_data(ctrl_employ_census,input_year=YEAR, exam_column=c)
        ctrl_report[ctrl_potential_indicators[c]].append(y)
ctrl_employ_census = ctrl_employ_census[ctrl_employ_census["Year"] == YEAR]
ctrl_SA2_employ_merge = pd.merge(SA2,ctrl_employ_census,left_on="SA2_CODE21",right_on="Code")
ctrl_SA2_employ = ctrl_SA2_employ_merge[ctrl_SA2_employ_merge.columns[:2]].copy()
for c in ctrl_SA2_employ_merge.columns[2:]:
    if c in ctrl_potential_indicators.keys():
        ctrl_SA2_employ.loc[:, ctrl_potential_indicators[c]] = ctrl_SA2_employ_merge[c].fillna(0).apply(convert_to_float)
ctrl_influence_system["employ"] = ctrl_SA2_employ.columns.tolist()[2:] 

# income
income_census = pd.read_csv("./data/all_income_census.csv")
income_census["Code"] = income_census["Code"].astype(str)
income_census["Year"] = income_census["Year"].astype(int)
for c in ctrl_potential_indicators.keys():
    if c in income_census.columns:
        income_census,y = fill_missing_data(income_census,input_year=YEAR, exam_column=c)
        ctrl_report[ctrl_potential_indicators[c]].append(y)
income_census = income_census[income_census["Year"] == YEAR]
ctrl_SA2_income_merge = pd.merge(SA2,income_census,left_on="SA2_CODE21",right_on="Code")#.fillna(0)
ctrl_SA2_income = ctrl_SA2_income_merge[ctrl_SA2_income_merge.columns[:2]].copy()
for c in ctrl_SA2_income_merge.columns[2:]:
    if c in ctrl_potential_indicators.keys():
        ctrl_SA2_income.loc[:, ctrl_potential_indicators[c]] = ctrl_SA2_income_merge[c].fillna(0).apply(convert_to_float)
ctrl_influence_system["income"] = ctrl_SA2_income.columns.tolist()[2:] 

# Xs 属性挂载
ctrl_SA2_Xs_Y = SA2.copy()
for c in ctrl_influence_system.keys():
    for i in ctrl_influence_system[c]:
        if i in ctrl_X_names:
            print(i)
            ctrl_SA2_Xs_Y.loc[:, i] = eval("ctrl_SA2_{}".format(c))[i]
# 分区统计
zs = ZonalStatistics(f'./data/VIIRS_Nighttime_{YEAR}_{CITY}.tif', SA2.to_crs(4326).copy())
zs.zonal_statistics(zone_field='NL', statistics_type='sum')
ctrl_SA2_Xs_Y.loc[:, "Urbanization level"] = zs.gdf["NL"] / (ctrl_SA2_Xs_Y.geometry.area/10**4)

ctrl_geo_df = ctrl_SA2_Xs_Y.copy()
ctrl_geo_df.loc[:,Y_name] = [j for j in mc.NaturalBreaks(ctrl_geo_df[i], k=3).yb]
# ---------------------------------------------------------------

ctrl_node_features = ctrl_SA2_Xs_Y[ctrl_X_names].values
ctrl_node_features = torch.tensor(ctrl_node_features, dtype=torch.float32)

# 对照组提取标签
ctrl_y = ctrl_geo_df[Y_name].values
ctrl_y = torch.tensor(ctrl_y, dtype=torch.long)

# 节点特征是x，边索引是edge_index
ctrl_x, ctrl_edge_index = ctrl_node_features, edge_index
# 应用高斯核函数
diff = ctrl_x[ctrl_edge_index[0]] - ctrl_x[ctrl_edge_index[1]]
ctrl_edge_attr = calculate_edge_attributes(ctrl_x, ctrl_edge_index)
# 将边属性添加到图数据对象中
ctrl_data = Data(x=ctrl_x, edge_index=edge_index, edge_attr=ctrl_edge_attr, y=ctrl_y)
ctrl_data.adjacency_dict = adjacency_dict

图中采样

In [ ]:
# 图中采样函数
def sampling(src_nodes, sample_num, neighbor_table):
    """根据源节点采样指定数量的邻居节点，注意使用的是有放回的采样；
    某个节点的邻居节点数量少于采样数量时，采样结果出现重复的节点

    Arguments:
        src_nodes {list, ndarray} -- 源节点列表
        sample_num {int} -- 需要采样的节点数
        neighbor_table {dict} -- 节点到其邻居节点的映射表

    Returns:
        np.ndarray -- 采样结果构成的列表
    """
    results = []
    for sid in src_nodes:
        # # 从节点的邻居中进行有放回地进行采样
        # res = np.random.choice(neighbor_table[sid], size=(sample_num,))
        # results.append(res)

        if len(neighbor_table[sid]) >= sample_num:
            res = np.random.choice(neighbor_table[sid], size=(sample_num,),replace=False)
        elif len(neighbor_table[sid]) > 0:
            res = np.random.choice(neighbor_table[sid], size=(sample_num,),replace=True)
        else:
            # If there are not enough neighbors, perform self-oversampling
            num_self_oversamples = sample_num - len(neighbor_table[sid])
            res = np.concatenate([neighbor_table[sid], np.full(num_self_oversamples, sid)])
            res = np.random.choice(res, size=(sample_num,), replace=False)
        results.append(res)
    return np.asarray(results).flatten()

def multihop_sampling(src_nodes, sample_nums, neighbor_table):
    """根据源节点进行多阶采样

    Arguments:
        src_nodes {list, np.ndarray} -- 源节点id
        sample_nums {list of int} -- 每一阶需要采样的个数
        neighbor_table {dict} -- 节点到其邻居节点的映射

    Returns:
        [list of ndarray] -- 每一阶采样的结果
    """
    sampling_result = [src_nodes]
    # print("sampling result = ", sampling_result)
    # print("sample_nums = ", sample_nums)

    for k, hopk_num in enumerate(sample_nums):
        # print("sampling_result[k] = ", sampling_result[k])
        hopk_result = sampling(sampling_result[k], hopk_num, neighbor_table)
        sampling_result.append(hopk_result)
    return sampling_result


图神经网络

In [ ]:
# 定义图神经网络模型
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
from tqdm import tqdm 

class NeighborAggregator(nn.Module):
    def __init__(self, input_dim, output_dim,
                 use_bias=False, aggr_method="mean"):
        """聚合节点邻居

        Args:
            input_dim: 输入特征的维度
            output_dim: 输出特征的维度
            use_bias: 是否使用偏置 (default: {False})
            aggr_method: 邻居聚合方式 (default: {mean})
        """
        super(NeighborAggregator, self).__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.use_bias = use_bias
        self.aggr_method = aggr_method
        self.weight = nn.Parameter(torch.Tensor(input_dim, output_dim))
        if self.use_bias:
            self.bias = nn.Parameter(torch.Tensor(self.output_dim))
        self.reset_parameters()

    def reset_parameters(self):
        init.kaiming_uniform_(self.weight)
        if self.use_bias:
            init.zeros_(self.bias)

    def forward(self, neighbor_feature):
        if self.aggr_method == "mean":
            aggr_neighbor = neighbor_feature.mean(dim=1)
        elif self.aggr_method == "sum":
            aggr_neighbor = neighbor_feature.sum(dim=1)
        elif self.aggr_method == "max":
            aggr_neighbor = neighbor_feature.max(dim=1)
        else:
            raise ValueError("Unknown aggr type, expected sum, max, or mean, but got {}"
                             .format(self.aggr_method))
        neighbor_hidden = torch.matmul(aggr_neighbor, self.weight)
        if self.use_bias:
            neighbor_hidden += self.bias

        return neighbor_hidden

    def extra_repr(self):
        return 'in_features={}, out_features={}, aggr_method={}'.format(
            self.input_dim, self.output_dim, self.aggr_method)


class GCN(nn.Module):
    def __init__(self, input_dim, hidden_dim,
                 activation=F.relu,
                 aggr_neighbor_method="mean",
                 aggr_hidden_method="sum"):
        """GCN层定义

        Args:
            input_dim: 输入特征的维度
            hidden_dim: 隐层特征的维度，
                当aggr_hidden_method=sum, 输出维度为hidden_dim
                当aggr_hidden_method=concat, 输出维度为hidden_dim*2
            activation: 激活函数
            aggr_neighbor_method: 邻居特征聚合方法，["mean", "sum", "max"]
            aggr_hidden_method: 节点特征的更新方法，["sum", "concat"]
        """
        super(GCN, self).__init__()
        assert aggr_neighbor_method in ["mean", "sum", "max"]
        assert aggr_hidden_method in ["sum", "concat"]
        self.embedding = None
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.aggr_neighbor_method = aggr_neighbor_method
        self.aggr_hidden_method = aggr_hidden_method
        self.activation = activation
        self.aggregator = NeighborAggregator(input_dim, hidden_dim,
                                             aggr_method=aggr_neighbor_method)
        self.weight = nn.Parameter(torch.Tensor(input_dim, hidden_dim))
        self.reset_parameters()

    def reset_parameters(self):
        init.kaiming_uniform_(self.weight)

    def forward(self, src_node_features, neighbor_node_features):
        neighbor_hidden = self.aggregator(neighbor_node_features)
        self_hidden = torch.matmul(src_node_features, self.weight)
        # print(f"src_node_features：{src_node_features.shape}")
        if self.aggr_hidden_method == "sum":
            hidden = self_hidden + neighbor_hidden
        elif self.aggr_hidden_method == "concat":
            hidden = torch.cat([self_hidden, neighbor_hidden], dim=1)
        else:
            raise ValueError("Expected sum or concat, got {}"
                             .format(self.aggr_hidden))
        if self.activation:
            if list(self.activation(hidden).shape)[0] == 189:
                self.embedding = self.activation(hidden)
            return self.activation(hidden)
        else:
            return hidden
        

    def extra_repr(self):
        output_dim = self.hidden_dim if self.aggr_hidden_method == "sum" else self.hidden_dim * 2
        return 'in_features={}, out_features={}, aggr_hidden_method={}'.format(
            self.input_dim, output_dim, self.aggr_hidden_method)


class GIGNN(nn.Module):
    def __init__(self, input_dim, hidden_dim,
                 num_neighbors_list):
        super(GIGNN, self).__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_neighbors_list = num_neighbors_list
        self.num_layers = len(num_neighbors_list)
        self.gcn = nn.ModuleList()
        self.gcn.append(GCN(input_dim, hidden_dim[0]))
        for index in range(0, len(hidden_dim) - 2):
            self.gcn.append(GCN(hidden_dim[index], hidden_dim[index + 1]))
        self.gcn.append(GCN(hidden_dim[0], hidden_dim[1], activation=None))

    def forward(self, node_features_list):
        # hidden = node_features_list
        # for l in range(self.num_layers):
        #     # 仅考虑一跳邻居
        #     src_node_features = hidden[l]
        #     src_node_num = len(src_node_features)
        #     neighbor_node_features = hidden[l + 1].view((src_node_num, self.num_neighbors_list[l], -1))
        #     # 调用当前层的 GCN
        #     h = self.gcn[l](src_node_features, neighbor_node_features)
        #     hidden.append(h)
        # # 返回最后一层的输出
        # return hidden[-1]
        hidden = node_features_list
        for l in range(self.num_layers):
            next_hidden = []
            gcn = self.gcn[l]
            for hop in range(self.num_layers - l):#
                # print(f"---第{l}层第{hop}跳---")
                src_node_features = hidden[hop]
                src_node_num = len(src_node_features)
                neighbor_node_features = hidden[hop+1].view((src_node_num, self.num_neighbors_list[hop], -1))
                h = gcn(src_node_features, neighbor_node_features)
                next_hidden.append(h)
            hidden = next_hidden
        return hidden[-1]

    def extra_repr(self):
        return 'in_features={}, num_neighbors_list={}'.format(
            self.input_dim, self.num_neighbors_list
        )

import os

class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        self.patience = patience
        self.counter = 0
        self.current_score = None
        self.delta = delta
        self.early_stop = False
        self.best_model_score = None  # 用于保存当前最优模型的指标值

    def __call__(self, f1_score, model, model_name="model", model_path="checkpoints/", message=""):
        checkpoint_path = os.path.join(model_path, '{}_checkpoint.pt'.format(model_name))
        global best_msg
        if os.path.exists(checkpoint_path):
            # 读取之前存储的最优模型的指标值
            checkpoint = torch.load(checkpoint_path)
            previous_best_score = checkpoint['best_model_score']
        else:
            previous_best_score = -float('inf')  # 未保存过模型时，默认为负无穷
        if self.best_model_score is None:
            self.best_model_score = previous_best_score  # 更新当前最优分数
            self.current_score = f1_score
            if f1_score > previous_best_score:
                # 覆盖最优模型检查点
                self.best_model_score = f1_score
                self.save_checkpoint(previous_best_score, f1_score, model, model_name, model_path, message)
            else:
                self.best_model_score = previous_best_score
            self.counter = 0
        elif f1_score < self.current_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            if f1_score > previous_best_score and f1_score != 1:
                # 覆盖最优模型检查点
                self.best_model_score = f1_score
                self.save_checkpoint(previous_best_score, f1_score, model, model_name, model_path, message)
            else:
                self.best_model_score = previous_best_score
            self.counter = 0
    def save_checkpoint(self, previous_best_score, f1_score, model, model_name, model_path, message):
        filename = '{}_checkpoint.pt'.format(model_name)
        # 保存模型的状态字典
        state = {
            'model_state_dict': model.state_dict(),
            'best_model_score': f1_score
        }
        torch.save(state, os.path.join(model_path, filename))
        print(message)
        global best_msg
        if model_name not in best_msg.keys():
            best_msg[model_name] = ""
        best_msg[model_name] = message
        print(f'F1 score increased ({previous_best_score:.6f} --> {f1_score:.6f}). Saving model ...')


在网络中增加权重

In [ ]:
# 增加权重

# 基于类别数量平衡权重
def calculate_class_weights_by_count(y):
    unique_classes, class_counts = np.unique(y, return_counts=True)
    total_samples = len(y)
    class_weights = total_samples / (len(unique_classes) * class_counts)
    return torch.tensor(class_weights, dtype=torch.float)

# 基于类别分布的统计信息
def calculate_class_weights_by_distribution(y):
    unique_classes, class_counts = np.unique(y, return_counts=True)
    total_samples = len(y)
    class_distribution = class_counts / total_samples
    class_weights = 1.0 / class_distribution
    return torch.tensor(class_weights, dtype=torch.float)

# 基于反类频率（Inverse Class Frequency）
def calculate_class_weights_by_inverse_frequency(y):
    unique_classes, class_counts = np.unique(y, return_counts=True)
    total_samples = len(y)
    class_distribution = class_counts / total_samples
    class_weights = 1.0 / class_distribution
    return torch.tensor(class_weights, dtype=torch.float)

# 基于Kappa系数
def calculate_class_weights_by_kappa(y, kappa_scores):
    unique_classes, class_counts = np.unique(y, return_counts=True)
    total_samples = len(y)
    class_weights = np.zeros(len(unique_classes))
    for i, c in enumerate(unique_classes):
        class_weights[i] = total_samples / (len(unique_classes) * class_counts[i]) * kappa_scores[i]
    return torch.tensor(class_weights, dtype=torch.float)

class_weights = calculate_class_weights_by_distribution(feature_source[Y_name].values)

print("Class Weights (by Count):", class_weights)

构建图神经网络模型

In [ ]:
# 数字与人类可读分类
# 5 类
# numberic_class = {
#     "Global": "Global",
#     "0": "Low Urbanization",
#     "1": "Emerging Urbanization",
#     "2": "Moderate Urbanization",
#     "3": "High Urbanization",
#     "4": "Intense Urbanization",
# }
# 3 类
numberic_class = {
    "Global": "Global",
    "0": "Low Urbanization",
    "1": "Moderate Urbanization",
    "2": "High Urbanization",
}
# 模型性能报文
reports_txt = {"Model":[], "Classification":[],  "F1":[], "Prec":[], "Recall":[]}
# 模型训练报文
train_txt = {}
# 模型最优训练效果
best_msg = {}

GIGNN 架构

加载模型检查点进行预测

In [ ]:
# 绘制预测与观测对比图
def predicted_plot(model_name, accuracy):

    import matplotlib.colors as mcolors
    import matplotlib.ticker as mticker
    import matplotlib as mpl
    import matplotlib.pyplot as plt

    mpl.rcParams['axes.unicode_minus'] = False
    plt.rcParams.update({'font.size': 22, 'font.family': "Arial"})
    # 创建子图
    fig, axes = plt.subplots(1, 3, figsize=(30, 20), dpi=300)

    # 绘制观察值
    ax = axes[0]
    geo_df.to_crs(4326).plot(column='Urbanization level', cmap='Blues', legend=False, edgecolor="#d9d9e3", ax=ax)
    ax.set_title(f"Observed ({YEAR})")
    set_axis_format(ax, 0.3, 0.15)  # 设置坐标轴格式

    # 绘制预测值
    ax = axes[1]
    SA2_predict = SA2.copy()
    SA2_predict.loc[:, "prediction"] = eval(f"{model_name}_predictions")
    SA2_predict.to_crs(4326).plot(column='prediction', cmap='Blues', legend=False, edgecolor="#d9d9e3", ax=ax)
    if_our = '(Ours)' if model_name == "GIGNN" else ""
    ax.set_title(f"Predicted by {model_name} {if_our} with accuracy {accuracy}")
    set_axis_format(ax, 0.3, 0.15)  # 设置坐标轴格式
    ax.set_ylabel("")
    # 创建颜色条
    cmap = plt.cm.Blues
    norm = mcolors.BoundaryNorm(np.arange(-0.5, 3), cmap.N)
    cbar_ax = fig.add_axes([0.15, 0.08, 0.7, 0.02])  # 定义颜色条位置
    cbar = fig.colorbar(
        plt.cm.ScalarMappable(norm=norm, cmap=cmap),
        cax=cbar_ax,
        boundaries=np.arange(-1, 3),
        ticks=[int(k) for k in keys_to_check],
        orientation='horizontal'
    )
    cbar.ax.set_xticklabels(list(numberic_class.values())[1:])  # 设置自定义标签

    ax = axes[2]
    # 创建冷暖色带，其中0为透明色
    cmap = mcolors.LinearSegmentedColormap.from_list(
        name='custom_cmap',
        colors=['white','pink','lightblue','#003462']
    )

    # 设置透明色
    cmap.set_bad(color='white', alpha=0)
    ax.set_title("Bias")
    set_axis_format(ax, 0.3, 0.15)  # 设置坐标轴格式
    ax.set_ylabel("")
    bias = geo_df[["geometry"]]
    bias.loc[:, "Bias Type"] = (geo_df["Urbanization level"] - SA2_predict["prediction"]).apply(lambda x: "Overestimated ({})".format(x) if x > 0 else ("Underestimated ({})".format(x) if x < 0 else "Correct")
    )
    bias.loc[:, "Validation"] = (geo_df["Urbanization level"].apply(lambda x: numberic_class[str(x)])+"-"+ SA2_predict["prediction"].apply(lambda x: numberic_class[str(x)]))
    bias.loc[:, "Bias"] = ((geo_df["Urbanization level"]-SA2_predict["prediction"]))
    bias.to_crs(4326).plot(ax=ax, column="Bias Type", cmap=cmap, aspect='auto', label="Bias level", legend=True)
    geo_df.to_crs(4326).plot(edgecolor="#d9d9e3", ax=ax,facecolor="none")

    # 保存并展示图形
    print(f'{CITY}_{YEAR}_Observed_Vs_Predicted_by_{model_name}')
    plt.savefig(f'./outputs/{CITY}_{YEAR}_Observed_Vs_Predicted_by_{model_name}.jpg', dpi=300, bbox_inches='tight', transparent=True)
    
    plt.show()

    return bias

In [ ]:
# GIGNN
# 获取要检查的键列表
keys_to_check = list(numberic_class.keys())[1:]  
# 加载模型权重
INPUT_DIM = target_data.num_features    # 输入维度
OUTPUT_DIM = len(np.unique(target_y))    # 输出维度
N = len(geo_df)
# Note: 采样的邻居阶数需要与GCN的层数保持一致
HIDDEN_DIM = [128, OUTPUT_DIM]   # 隐藏单元节点数
NUM_NEIGHBORS_LIST = [6, 12]   # 每阶采样邻居的节点数 [6,6]
assert len(HIDDEN_DIM) == len(NUM_NEIGHBORS_LIST)
BTACH_SIZE = 64    # 批处理大小 128
EPOCHS = 1000
NUM_BATCH_PER_EPOCH = 1    # 每个epoch循环的批次数
LEARNING_RATE = 0.01    # 学习率
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

GIGNN_model = GIGNN(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM,
                num_neighbors_list=NUM_NEIGHBORS_LIST).to(DEVICE)
checkpoint = torch.load("./checkpoints/GIGNN_checkpoint.pt")
GIGNN_model.load_state_dict(checkpoint['model_state_dict'])
# 将模型设置为评估模式
GIGNN_model.eval()

# 使用模型进行预测
GIGNN_predictions = []
GIGNN_accuracy = 0
with torch.no_grad():
    bsi = np.where(torch.ones(N, dtype=torch.bool))[0]
    pre_sampling_result = multihop_sampling(bsi, NUM_NEIGHBORS_LIST, target_data.adjacency_dict)
    pre_x = [target_x[idx].float().to(DEVICE) for idx in pre_sampling_result]
    output = GIGNN_model(pre_x)
    pred = output.argmax(dim=1)  # 获取预测的类别
    GIGNN_predictions.extend([p for p in pred.tolist()])
    accuracy = torch.eq(pred.to(DEVICE), target_data.y.to(DEVICE)).float().mean().item()
    GIGNN_accuracy = round(accuracy,3)
GIGNN_bias = predicted_plot(model_name="GIGNN",accuracy=GIGNN_accuracy)

In [ ]:
# GAT
class GAT(torch.nn.Module):
    def __init__(self):
        super(GAT, self).__init__()
        self.conv1 = GATConv(ctrl_data.num_features, 16)
        self.conv2 = GATConv(16, len(np.unique(ctrl_data.y)))
        self.dropout = torch.nn.Dropout(p=0.5)
        self.batchnorm1 = torch.nn.BatchNorm1d(16)
        self.batchnorm2 = torch.nn.BatchNorm1d(len(np.unique(ctrl_data.y)))
    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.batchnorm1(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = self.batchnorm2(x)
        return x

GAT_model = GAT().to("cpu")
checkpoint = torch.load("./checkpoints/GAT_checkpoint.pt")
GAT_model.load_state_dict(checkpoint['model_state_dict'])
# 将模型设置为评估模式
GAT_model.eval()
# 使用模型进行预测
GAT_predictions = []
GAT_accuracy = 0
with torch.no_grad():
    out = GAT_model(ctrl_data)
    _, pred = torch.max(out, dim=1)
    GAT_predictions.extend([p for p in pred.tolist()])
    correct = (pred == ctrl_data.y).sum().item()
    GAT_accuracy = round(correct / ctrl_data.num_nodes, 3)
GAT_bias = predicted_plot(model_name="GAT",accuracy=GAT_accuracy)

In [ ]:
# GIN
class GIN(torch.nn.Module):
    def __init__(self):
        super(GIN, self).__init__()
        nn1 = Sequential('x, edge_index', [(GINConv(nn.Linear(ctrl_data.num_features, 16)), 'x, edge_index -> x')])
        nn2 = Sequential('x, edge_index', [(GINConv(nn.Linear(16, len(np.unique(ctrl_data.y.cpu())))).cpu(), 'x, edge_index -> x')])
        self.conv1 = nn1
        self.conv2 = nn2
        self.dropout = torch.nn.Dropout(p=0.5)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

GIN_model = GIN().to("cpu")
checkpoint = torch.load("./checkpoints/GIN_checkpoint.pt")
GIN_model.load_state_dict(checkpoint['model_state_dict'])
# 将模型设置为评估模式
GIN_model.eval()
# 使用模型进行预测
GIN_predictions = []
GIN_accuracy = 0
with torch.no_grad():
    out = GIN_model(ctrl_data.to("cpu"))
    _, pred = torch.max(out, dim=1)
    GIN_predictions.extend([p for p in pred.tolist()])
    correct = (pred == ctrl_data.y).sum().item()
    GIN_accuracy = round(correct / ctrl_data.num_nodes, 3)
GIN_bias = predicted_plot(model_name="GIN",accuracy=GIN_accuracy)

In [ ]:
# ChebNet
class ChebNet(torch.nn.Module):
    def __init__(self):
        super(ChebNet, self).__init__()
        self.conv1 = ChebConv(ctrl_data.num_features, 16, K=2)
        self.conv2 = ChebConv(16, len(np.unique(ctrl_data.y)), K=2)
        self.dropout = torch.nn.Dropout(p=0.5)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index

        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)

        return F.log_softmax(x, dim=1)

ChebNet_model = ChebNet().to("cpu")
checkpoint = torch.load("./checkpoints/ChebNet_checkpoint.pt")
ChebNet_model.load_state_dict(checkpoint['model_state_dict'])
# 将模型设置为评估模式
ChebNet_model.eval()
# 使用模型进行预测
ChebNet_predictions = []
ChebNet_accuracy = 0
with torch.no_grad():
    out = ChebNet_model(ctrl_data.to("cpu"))
    _, pred = torch.max(out, dim=1)
    ChebNet_predictions.extend([p for p in pred.tolist()])
    correct = (pred == ctrl_data.y).sum().item()
    ChebNet_accuracy = round(correct / ctrl_data.num_nodes, 3)
    GIN_bias = predicted_plot(model_name="ChebNet", accuracy=ChebNet_accuracy)

In [ ]:
# MPNN
class MPNN(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super(MPNN, self).__init__(aggr='add')  # "Add" aggregation.
        self.lin = torch.nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index):
        # x has shape [N, in_channels]
        # edge_index has shape [2, E]

        # Step 1: Add self-loops to the adjacency matrix.
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))

        # Step 2: Linearly transform node feature matrix.
        x = self.lin(x)

        # Step 3: Compute normalization.
        row, col = edge_index
        deg = degree(row, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # Step 4: Start propagating messages.
        x = self.propagate(edge_index, size=(x.size(0), x.size(0)), x=x, norm=norm)

        return F.log_softmax(x, dim=1)

    def message(self, x_j, norm):
        # x_j has shape [E, out_channels]

        # Step 5: Normalize node features.
        return norm.view(-1, 1) * x_j

MPNN_model = MPNN(ctrl_data.num_features, len(np.unique(ctrl_data.y))).to("cpu")
checkpoint = torch.load("./checkpoints/MPNN_checkpoint.pt")
MPNN_model.load_state_dict(checkpoint['model_state_dict'])
# 将模型设置为评估模式
MPNN_model.eval()
# 使用模型进行预测
MPNN_predictions = []
MPNN_accuracy = 0
with torch.no_grad():
    out = MPNN_model(ctrl_data.x, ctrl_data.edge_index)
    _, pred = torch.max(out, dim=1)
    MPNN_predictions.extend([p for p in pred.tolist()])
    correct = (pred == ctrl_data.y).sum().item()
    MPNN_accuracy = round(correct / ctrl_data.num_nodes, 3)
    MPNN_bias = predicted_plot(model_name="MPNN", accuracy=MPNN_accuracy)

In [ ]:
# RF
import joblib
from sklearn.metrics import accuracy_score
# 加载模型
RF_model = joblib.load('./checkpoints/RF.joblib')
RF_predictions = RF_model.predict(ctrl_data.x)
RF_accuracy = round(accuracy_score(ctrl_data.y, RF_predictions), 3)
RF_bias = predicted_plot(model_name="RF", accuracy=RF_accuracy)

In [ ]:
# auc = len(bias[bias["Bias"]==0])/len(bias)*100
# report["accuracy"].append(auc)
# print(f"{CITY} ({YEAR}) | Accuracy：{round(auc,2)}%")



In [ ]:
auc = len(GIGNN_bias[GIGNN_bias["Bias"]==0])/len(bias)*100
report["accuracy"].append(auc)
print(f"{CITY} ({YEAR}) | Accuracy：{round(auc,2)}%")
GIGNN_bias[GIGNN_bias["Bias"]!=0].to_file(f"./outputs/bias_{CITY}_{YEAR}.shp", driver="ESRI Shapefile")
pd.value_counts(GIGNN_bias["Bias Type"])

In [ ]:
print("Validation（左侧为观测值，右侧为预测值）")
pd.value_counts(GIGNN_bias[GIGNN_bias["Validation"].apply(lambda x: x.split("-")[0] != x.split("-")[1])]["Validation"])

In [ ]:
valid_models = ["GIGNN", "GAT", "GIN", "ChebNet", "MPNN", "RF"]
valid_accuracy = []
for m in valid_models:
    valid_accuracy.append(eval(f"{m}_accuracy"))
valid_df = pd.DataFrame({
    "Year": [YEAR for i in range(len(valid_accuracy))], 
    "Model": valid_models,
    "Accuracy": valid_accuracy,
})
valid_df

In [ ]:
# 绘制水平柱状图
plt.figure(figsize=(16, 16))

# 绘制带有颜色的条形图
ax = valid_df[["Model", "Accuracy"]].plot(kind="bar",x="Model", y="Accuracy",colormap="coolwarm_r", width=0.35,figsize=(12, 6), alpha=0.6)
# plt.title(f"Accuracy comparison for predicting urbanization level in {YEAR} between models",fontdict={"size":24})
plt.xlabel("Model")
plt.ylabel("Accuracy")
plt.ylim(0,1.15)
plt.legend(loc="upper right", ncol=6)
plt.xticks(rotation=0) #range(len(valid_df)), [f'{valid_df.loc[i,"Model"]}\n({valid_df.loc[i,"Accuracy"]})' for i in valid_df.index], 
for idx, p in enumerate(ax.patches):
    ax.annotate(f'{p.get_height():.3f}', (p.get_x() + p.get_width() / 2., p.get_height()),#p.get_height()
                ha='center', va='center', xytext=(0, 10), textcoords='offset points')
plt.savefig('./outputs/{}.png'.format(f"Accuracy comparison for predicting urbanization level in {YEAR} between models"), dpi=300, bbox_inches='tight', transparent=True)
plt.show()

In [ ]:
valid_df.to_clipboard(index=None)

In [ ]:
report_df = pd.DataFrame(report)
report_df["accuracy"] = report_df["accuracy"].apply(lambda x: round(x/100, 3))
report_df

In [ ]:
report

In [ ]:
report_df.T

In [ ]:
tmp_report = classification_report(geo_df["Urbanization level"], SA2_predict["prediction"], output_dict=True, zero_division=1)
print("GIGNN Performance Metrics")
tmp_report = pd.DataFrame(tmp_report).rename(columns=numberic_class)
tmp_report[tmp_report.columns[:-2]]

input 5 -> hiden 128 -> output 3